# DuckDB — User Journey

DuckDB is a SQL layer directly on top of the Parquet files in S3. No ingestion step — it reads the files in place via `httpfs`.

**Steps:** access → query → filtered query → append → evolve schema

In [1]:
import time
from contextlib import contextmanager

timings = {}

@contextmanager
def bench(step):
    """Time a step and record it for the final benchmark table."""
    t0 = time.perf_counter()
    yield
    timings[step] = time.perf_counter() - t0
    print(f"  {step}: {timings[step]:.3f}s")

## 1. Access

Resolve the artifacts to their `s3://` paths and register a view. DuckDB reads directly from S3 — nothing is downloaded.

In [ ]:
import lamindb as ln
import duckdb
import pandas as pd

ln.track("jWu2DN5COpfJ", project="Lakehouse benchmarks v1")
collection = ln.Collection.get("K6X8Ejk3fjgAZT6h0000")
s3_paths = [str(a.path) for a in collection.ordered_artifacts.all()]

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

with bench("access"):
    con.execute(f"CREATE OR REPLACE VIEW cnv_vcf AS SELECT * FROM read_parquet({s3_paths})")
    # Force a real read so the timer means something
    _count = con.execute("SELECT COUNT(*) FROM cnv_vcf").fetchone()[0]

print(f"Total rows: {_count:,}")


→ connected lamindb: laminlabs/lakehouse-benchmarks


d:\Raaghav\Lamin\lakehouse-benchmarks\.venv\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


→ loaded Transform('jWu2DN5COpfJ0001', key='duckdb_pipeline.ipynb'), re-started Run('jfyy4vF17U7pRBMO') at 2026-06-13 13:47:10 UTC
→ notebook imports: duckdb==1.5.3 lamindb-core==2.5.1 pandas==2.3.3
• recommendation: to identify the notebook across renames, pass the uid: ln.track("jWu2DN5COpfJ", project="Lakehouse benchmarks v1")
  access: 3.229s
Total rows: 8,929


## 2. Query — per-sample stats and recurrent regions (pure SQL)

In [3]:
with bench("query_stats"):
    stats_df = con.execute("""
        SELECT
            SAMPLE_NAME                                            AS Sample,
            COUNT(*)                                               AS Total_CNVs,
            COUNT(*) FILTER (WHERE INFO_SVLEN < 0)                 AS Deletions,
            MEDIAN(ABS(INFO_SVLEN)) FILTER (WHERE INFO_SVLEN < 0)  AS Median_Deletion_Size,
            COUNT(*) FILTER (WHERE SAMPLE_GT = '1/1')              AS Homozygous_CNVs,
            COUNT(*) FILTER (WHERE SAMPLE_GT = '0/1')              AS Heterozygous_CNVs
        FROM cnv_vcf
        GROUP BY SAMPLE_NAME
    """).df()
stats_df.head()

  query_stats: 2.234s


,Sample,Total_CNVs,Deletions,Median_Deletion_Size,Homozygous_CNVs,Heterozygous_CNVs
0,HG00101,1503,536,3286.5,135,382
1,HG00102,1491,537,3800.0,164,373
2,HG00100,1412,516,3997.0,146,370
3,HG00096,1499,523,3323.0,123,381
4,HG00097,1488,547,3614.0,154,393


In [4]:
with bench("query_recurrent"):
    recurrent = con.execute("""
        SELECT CHROM || ':' || CAST((POS // 1000) * 1000 AS VARCHAR) AS region_key,
               COUNT(DISTINCT SAMPLE_NAME) AS sample_count
        FROM cnv_vcf
        GROUP BY region_key
        HAVING COUNT(DISTINCT SAMPLE_NAME) >= 2
        ORDER BY sample_count DESC
    """).df()
print(f"Identified {len(recurrent)} recurrent regions.")

  query_recurrent: 2.589s
Identified 1903 recurrent regions.


## 3. Filtered query — predicate pushdown reads only relevant row groups

In [5]:
with bench("filtered_query"):
    filtered = con.execute("""
        SELECT * FROM cnv_vcf
        WHERE CHROM = '1' AND POS BETWEEN 1000000 AND 50000000
    """).df()
print(f"Variants in chr1:1M-50M: {len(filtered)}")

  filtered_query: 1.566s
Variants in chr1:1M-50M: 0


## 4. Append — extend the path list and redefine the view

No transaction guarantees, but zero overhead — the view just points at more files.

In [6]:
# new_path = str(new_artifact.path)  # an appended LaminDB artifact on S3
# s3_paths.append(new_path)
with bench("append"):
    con.execute(f"CREATE OR REPLACE VIEW cnv_vcf AS SELECT * FROM read_parquet({s3_paths})")
print(f"Rows after append: {con.execute('SELECT COUNT(*) FROM cnv_vcf').fetchone()[0]:,}")

  append: 0.756s
Rows after append: 8,929


## 5. Evolve schema — view redefinition in SQL

In [7]:
with bench("evolve_schema"):
    con.execute(f"""
        CREATE OR REPLACE VIEW cnv_vcf AS
        SELECT *, NULL::BOOLEAN AS QC_PASS FROM read_parquet({s3_paths})
    """)
print("View now exposes QC_PASS column")

  evolve_schema: 0.733s
View now exposes QC_PASS column


## Benchmark summary

In [8]:
import pandas as pd
pd.DataFrame(
    [{"step": k, "seconds": round(v, 3)} for k, v in timings.items()]
)

,step,seconds
0,access,3.229
1,query_stats,2.234
2,query_recurrent,2.589
3,filtered_query,1.566
4,append,0.756
5,evolve_schema,0.733


In [10]:
ln.finish()


d:\Raaghav\Lamin\lakehouse-benchmarks\.venv\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


→ finished Run('jfyy4vF17U7pRBMO') after 1m at 2026-06-13 13:48:16 UTC
→ go to: https://lamin.ai/laminlabs/lakehouse-benchmarks/transform/jWu2DN5COpfJ0001
→ to update your notebook from the CLI, run: lamin save duckdb_pipeline.ipynb
